# Path of Exile 2 text SFT with Unsloth

This notebook is a **supervised fine-tune (SFT)** of Llama 3.1 8B Instruct on Path of Exile 2 crafting Q&A. It is written as a class: every code cell is preceded by a lesson that explains *why* the code exists, not only *what* it does.

You do not need prior fine-tuning experience. Read the markdown, run the cell, then read the next markdown which interprets the output you just saw.

## What problem are we solving?

A pretrained LLM already "knows" a lot of internet text. It does **not** automatically answer in *your* style, with *your* facts, for *your* task. Fine-tuning updates the model so that given a PoE 2 crafting scenario, it replies like the guides in `poe2_data.jsonl`.

This is **not** RAG (retrieval). RAG stuffs extra documents into the prompt at inference time and leaves weights unchanged. Fine-tuning **changes weights**. Unsloth's docs put it plainly: fine-tuning can inject knowledge and change behavior; RAG cannot change the model itself.

## The three layers of "training" people confuse

1. **Pretraining** — predict the next token on a huge crawl of text. Produces a *base* model. Expensive. We are not doing this.
2. **Supervised fine-tuning (SFT)** — show the model many `(prompt, ideal answer)` pairs and train it to emit the answer. That is **this notebook**.
3. **Preference / RL methods** (DPO, GRPO, …) — later stage. The model generates answers and is scored. Skip this until SFT works.

## Instruct vs base (this choice matters)

| Kind | Typical name | Best for |
| --- | --- | --- |
| **Base** | `Llama-3.1-8B` | Continued pretraining, or SFT from scratch on a custom template |
| **Instruct** | `Llama-3.1-8B-Instruct` | Chat / Q&A. Already trained to follow user/assistant turns |

Unsloth recommends starting with an **Instruct** model. Instruct models already speak chat-template language, so you need **less data**. This notebook uses `Meta-Llama-3.1-8B-Instruct`.

## Full fine-tune vs LoRA vs QLoRA

Llama 3.1 8B has ~8 billion parameters. Updating all of them (full fine-tune) needs a lot of VRAM and is usually unnecessary.

**LoRA** (Low-Rank Adaptation): freeze the original weights \(W\). Beside each big matrix, train two thin matrices \(A\) and \(B\) so the effective weight is:

\[
\hat{W} = W + \frac{\alpha}{r} AB
\]

You typically train ~0.5–1% of parameters. Quality can match full fine-tuning when LoRA is applied to **all major linear layers**.

**QLoRA**: same LoRA adapters, but the frozen base is stored in **4-bit**. That cuts VRAM by ~4×. Slightly slower / a hair less accurate than 16-bit LoRA. Unsloth's **dynamic 4-bit** checkpoints (`*-unsloth-bnb-4bit`) recover most of that accuracy gap.

This notebook is **QLoRA**: `load_in_4bit=True` + LoRA adapters.

Unsloth's rule of thumb: try LoRA/QLoRA first. If it fails, full fine-tuning almost never magically fixes a bad dataset.

## What Unsloth is doing for you

[Unsloth](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide) is a training stack on top of Hugging Face `transformers` + TRL. It:

- patches Llama kernels for faster forward/backward
- loads 4-bit models with less VRAM
- provides chat-template helpers (`get_chat_template`, `train_on_responses_only`)
- can export LoRA adapters or GGUF for llama.cpp / Ollama

The training *recipe* is still standard SFT. Unsloth makes that recipe fit on a single consumer GPU (this machine: RTX 4070 Ti SUPER, 16 GB).

## Pipeline (keep this map in your head)

```
1. Load 4-bit Instruct model + tokenizer
2. Attach LoRA adapters (the only weights we train)
3. Convert Alpaca JSONL → Llama 3.1 chat text
4. Split train / eval
5. SFTTrainer, loss only on assistant tokens
6. Train ~2 epochs
7. Generate a test answer
8. Save LoRA (required). Optional: merge + GGUF
```

## Dataset

`poe2_data.jsonl` — 260 Alpaca-style rows:

- `instruction` — the task ("explain this omen")
- `input` — the crafting scenario (base item, goal, inventory)
- `output` — the gold crafting guide

That is a **small** domain dataset. Small data is fine for *style and format* on an Instruct model. It will not teach the model the entire PoE 2 wiki. If answers look generic after training, the fix is usually **better / more data**, not a bigger rank.

Official references used throughout these notes:

- [Fine-tuning LLMs Guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide)
- [LoRA Hyperparameters Guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/lora-hyperparameters-guide)
- [Chat Templates](https://unsloth.ai/docs/basics/chat-templates)
- [Datasets Guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide/datasets-guide)
- [Saving to GGUF](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)


## 1. Load the 4-bit Instruct model

### Why this cell exists

Training needs two objects in GPU memory:

1. **`model`** — the neural net (transformer blocks, attention, MLPs).
2. **`tokenizer`** — the text ↔ integer converter. Models do not see words. They see **token IDs**. `"Hello"` might become `[9906]`. The tokenizer also owns the **chat template** (how user/assistant turns are wrapped in special tokens).

`FastLanguageModel.from_pretrained` is Unsloth's loader. Use it instead of raw `AutoModelForCausalLM.from_pretrained` so Unsloth can patch kernels and apply 4-bit loading correctly. Import Unsloth **before** heavy `transformers` use; the first import prints `Will patch your computer...`.

### Line by line

**`from unsloth import FastLanguageModel`**  
Entry point for load, LoRA, inference mode, and (later) GGUF export.

**`import torch`**  
PyTorch. We use it for CUDA device queries. Unsloth uses it under the hood for tensors and autograd.

**`max_seq_length = 2048`**  
Maximum tokens in one training example (prompt + answer). Llama 3.1 can do 128k context, but 2048 is Unsloth's recommended starting point: cheaper, faster, enough for these crafting guides. If a row is longer, it is **truncated**. If you later see chopped answers, raise this (VRAM goes up). Unsloth can train longer context than stock Hugging Face on the same GPU.

**`dtype = None`**  
Compute dtype for the unfrozen / LoRA math. `None` means auto:

- **bfloat16** on Ampere and newer (this 4070 Ti SUPER: yes — you will see `Bfloat16 = TRUE`)
- **float16** on older GPUs

Do not pick `float32` for fine-tuning; it wastes VRAM. bf16 is preferred when available because it does not need a loss-scaling dance like fp16.

**`load_in_4bit = True`**  
This **is** QLoRA. The frozen 8B weights live in 4-bit. Set this `False` (or `load_in_16bit=True`) for 16-bit LoRA: more VRAM, slightly more accuracy. Unsloth: only one of 4-bit / 8-bit / 16-bit / full FT should be on.

**`model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"`**  
Hugging Face repo id. Unsloth naming:

| Suffix | Meaning |
| --- | --- |
| `unsloth-bnb-4bit` | Unsloth **dynamic 4-bit**. A bit more VRAM than plain bnb-4bit, much closer to 16-bit accuracy |
| `bnb-4bit` (no `unsloth`) | Standard BitsAndBytes 4-bit |
| no suffix | Original 16-bit (or 8-bit) weights |

**Instruct** in the name means chat-tuned. A smaller/faster option in the comment is `unsloth/Llama-3.2-3B-Instruct` if 8B is tight on VRAM.

**`FastLanguageModel.from_pretrained(...)`**  
Downloads weights if needed, materializes the 4-bit model on GPU, returns `(model, tokenizer)`. The banner you see lists GPU, CUDA capability, Torch version, and whether Flash Attention is on. `FA2 = False` here is fine; Unsloth still patches Llama.

### What "2x faster free finetuning" means

Unsloth replaces some PyTorch autograd paths with hand-written backward kernels and memory tricks (gradient offload, checkpointing). Same math, less time and VRAM. You do not configure that beyond using their API.

### Common mistakes at this step

- Loading a **Vision** checkpoint for text-only data (this notebook used to). Text SFT wants a text Instruct model.
- Forgetting `max_seq_length` — later trainer and loader must agree.
- Training 4-bit then serving 16-bit (or the reverse) without care. Unsloth: *train and serve in the same precision when you can.*


In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto: bf16 on Ampere+ (this GPU), else fp16
load_in_4bit = True

# Dynamic 4-bit Instruct checkpoint (higher accuracy than plain bnb-4bit).
# Smaller/faster option: "unsloth/Llama-3.2-3B-Instruct"
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/eplinux/unsloth-ft/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4070 Ti SUPER. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/home/eplinux/unsloth-ft/.venv/lib/python3.13/site-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 88.27it/s] 
Unsloth: Will load unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


### What you should have just seen

The Unsloth banner confirms:

- **Unsloth 2026.x + Transformers** — stack versions.
- **NVIDIA GeForce RTX 4070 Ti SUPER, 16 GB** — one GPU.
- **CUDA 8.9, Bfloat16 = TRUE** — this GPU can use bf16. Good.
- **Fast Llama patching** — kernels are hooked.
- Weight load ~291 shards, a few seconds if cached.

Warnings (`IProgress`, `HF_HUB_ENABLE_HF_TRANSFER`, unauthenticated Hub) are noisy but not fatal. A `HF_TOKEN` only helps rate limits.

---

## 1b. Check VRAM before you attach LoRA

### Why this cell exists

Fine-tuning dies in two boring ways: **OOM** (out of memory) and **you thought you had headroom**. After the 4-bit load, measure what is left **before** LoRA, activations, and the optimizer.

`torch.cuda` talks to the driver:

- **`get_device_name(0)`** — GPU 0's marketing name.
- **`get_device_properties(0).total_memory`** — total VRAM in bytes. Divide by `1024**3` for GiB. Expect ~16.0 on this card.
- **`mem_get_info()[0]`** — **free** bytes right now (tuple is `(free, total)`).

After loading 8B QLoRA, seeing on the order of **~9 GB free** of 16 GB is normal: the 4-bit base plus runtime ate several GB. LoRA adapters are small; the hungry parts during training are:

- activations (grows with batch size and `max_seq_length`)
- AdamW states (2× trainable params, reduced here by `adamw_8bit`)
- logits for 128k vocab

If free VRAM were ~1–2 GB already, you would drop `per_device_train_batch_size` to 1, shorten `max_seq_length`, or switch to the 3B Instruct model **before** training, not after OOM.


In [2]:
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"Free: {torch.cuda.mem_get_info()[0] / 1024**3:.1f} GB")


NVIDIA GeForce RTX 4070 Ti SUPER
VRAM: 16.0 GB
Free: 9.0 GB


### How to read the VRAM print

You should see the 4070 Ti SUPER, **16.0 GB** total, and free memory **after** the 4-bit load (previously ~9 GB). That remaining budget is what training will live in.

---

## 2. Attach LoRA adapters (the only weights we train)

### Why this cell exists

The 8B weights are frozen in 4-bit. If we trained now, nothing useful would update. `get_peft_model` injects **LoRA** layers (PEFT = Parameter-Efficient Fine-Tuning). After this, `model` is a PEFT model: forward = frozen \(W\) + \(AB\) adapters.

Unsloth's print `patched 32 layers with 32 QKV ...` means all 32 Llama 3.1 blocks got Q/K/V, output projection, and MLP adapters. Llama 3.1 8B has 32 transformer layers.

### LoRA in one picture

Each target linear layer of shape \(d_{\text{out}} \times d_{\text{in}}\) would normally have that many trainable weights. LoRA uses:

- \(A\): \(r \times d_{\text{in}}\)
- \(B\): \(d_{\text{out}} \times r\)

with rank **`r` ≪ d**. For `r=16` this is tiny vs a 4096×4096 matrix. After training, this notebook reported **41,943,040 trainable params of 8,072,204,288 (0.52%)**. That is the whole point of LoRA.

### Every argument

**`r=16`** — LoRA rank. Unsloth suggested values: 8, 16, 32, 64, 128. Higher \(r\) = more capacity, more VRAM, easier overfitting. For 260 crafting rows, **16 is the right default**. Jumping to 128 on this dataset is how you memorize the JSONL.

**`target_modules`** — which linear layers get adapters.

| Module | Where | Role |
| --- | --- | --- |
| `q_proj`, `k_proj`, `v_proj`, `o_proj` | Attention | How tokens look at each other |
| `gate_proj`, `up_proj`, `down_proj` | MLP / FFN | Feed-forward transform per token |

QLoRA paper + Unsloth: target **all seven**. Attention-only or MLP-only is worse. Dropping modules to save VRAM saves almost nothing and costs quality.

**`lora_alpha=16`** — scales the update: \(\alpha / r\). Unsloth: set \(\alpha = r\) or \(\alpha = 2r\). Here \(\alpha = r = 16\), so scale = 1. If the finetune is too shy, try `lora_alpha=32`. If it is too specialized, they even suggest scaling alpha **down after** training.

**`lora_dropout=0`** — randomly zero LoRA activations. Unsloth optimizes the `0` path (faster). For short SFT runs dropout is a weak regularizer. If you overfit, try `0.05–0.1` before you panic-change rank.

**`bias="none"`** — do not train bias vectors. Faster, less memory, no real quality win from training them.

**`use_gradient_checkpointing="unsloth"`** — do not store every activation for backward; recompute them. Classic checkpointing saves VRAM; Unsloth's extra path claims ~30% more savings and long-context friendliness. Use `"unsloth"` unless you are debugging speed.

**`random_state=3407`** — seed for LoRA init (and later dataset split uses the same number). Same seed ⇒ comparable runs. 3407 is Unsloth's notebook convention, not magic.

**`use_rslora=False`** — rank-stabilized LoRA scales by \(\alpha / \sqrt{r}\) instead of \(\alpha / r\). Sometimes helps at **high** rank. Leave off at r=16.

**`loftq_config=None`** — LoftQ initializes \(A,B\) from SVD of \(W\). Can help accuracy; spikes VRAM at start. Beginners: leave `None`.

### What we are *not* doing

- Not training embeddings or lm_head (unless you add special tokens — must happen **before** `get_peft_model`).
- Not full fine-tune (`full_finetuning=True`).
- Not changing the 4-bit frozen weights; only adapters.

After this cell the model is ready to train. Next we must feed it text in the **exact chat format** Llama 3.1 Instruct expects.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


Unsloth 2026.9.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


### What you should have just seen

`Unsloth ... patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.` All attention + MLP projections in all 32 blocks have LoRA. If that number were smaller, `target_modules` would be wrong.

---

## 3. Dataset: Alpaca JSONL → Llama 3.1 chat `text`

### Why this cell exists

The trainer does not understand "instruction / input / output" columns. It needs **one string per row** that looks like the model at inference: special tokens, user turn, assistant turn. Wrong template ⇒ the model learns a format it will never see in production (or you get gibberish in Ollama later).

Unsloth's datasets guide: apply `get_chat_template`, map conversations with `tokenizer.apply_chat_template`, store the result in a `"text"` column. This notebook **precomputes** that column. It does **not** pass a live `formatting_func` into the trainer (easier to debug: you can print `text`).

### Alpaca JSONL (what is on disk)

Each line of `poe2_data.jsonl` is JSON:

```json
{
  "instruction": "Provide a step-by-step PoE 2 crafting guide ...",
  "input": "Crafting Item: Omen of Connections\\n...",
  "output": "### Path of Exile 2 Crafting Overview: ..."
}
```

- **instruction** — the user intent / task wording.
- **input** — extra context (item, goal). May be empty on some datasets; here it is the scenario.
- **output** — the answer we want the model to produce.

This is **single-turn** SFT: one user message, one assistant message. Multi-turn chat would be a list of several role turns; Unsloth can stitch Alpaca rows into fake multi-turn via `conversation_extension`, which we are **not** using.

### Hugging Face `Dataset`

`Dataset.from_list(rows)` builds an in-memory Arrow table. `.map(fn)` applies a function to every row (or batch). That is the standard HF datasets workflow; Unsloth does not replace it.

### `get_chat_template(tokenizer, chat_template="llama-3.1")`

Unsloth **replaces / fixes** the tokenizer chat template. Upstream templates are sometimes wrong; Unsloth maintains corrected ones (`llama-3.1`, `chatml`, `gemma-3`, …). Always match the template to the model family. Gemma tokens on a Llama model = broken training.

Llama 3.1 turns look like:

```
<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>

...optional system...
<|eot_id|>
<|start_header_id|>user<|end_header_id|>

USER TEXT<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>

ASSISTANT TEXT<|eot_id|>
```

`<|eot_id|>` is end-of-turn (the model should stop). If you train without it, generations ramble.

### `alpaca_to_conversation`

Builds the ChatML-style list Unsloth expects: `role` / `content` (not ShareGPT `from` / `value`). If you had ShareGPT data you would call `standardize_sharegpt` first.

User content = `instruction`, or `instruction + "\n\n" + input` when input is non-empty. That is the usual Alpaca merge: the model should see both the task and the scenario in **one** user turn.

Assistant content = `output` unchanged.

### `formatting_prompts_func` (batched)

For each conversation:

```python
tokenizer.apply_chat_template(
    convo,
    tokenize=False,              # return a string, not ids (trainer tokenizes later)
    add_generation_prompt=False, # include the assistant answer; do NOT append an empty assistant header
)
```

**`add_generation_prompt`** is the training vs inference switch:

| Stage | Value | Effect |
| --- | --- | --- |
| Training | `False` | String contains the gold assistant reply + `<|eot_id|>` |
| Inference | `True` | String ends at the assistant header so the model **fills in** the reply |

If you train with `True`, you would **drop the answers** from the loss targets. If you infer with `False`, the template may not open the assistant turn.

The function returns `{"text": texts}` — new column named `text`. That name must match `dataset_text_field="text"` later.

### Train / eval split

```python
dataset.train_test_split(test_size=0.2, seed=3407)
```

260 rows → **208 train / 52 eval** (80/20). Same seed as LoRA so the split is reproducible.

Unsloth: if you already burned the whole set on training, you can only **manually** judge quality. Holding out 20% lets `eval_strategy="epoch"` compute a held-out loss. 52 rows is small; treat eval loss as a trend, not a scientific benchmark.

### Why 260 rows can still work

Instruct models already know English and chat structure. You are steering **domain format** (PoE 2 omen/currency guides) and a bit of local jargon. That is a different job from teaching a base model to speak. Quality of `output` text matters more than clever LoRA settings.

Run the cell, then look at `conversations` (Python dicts) in the printout. The next cell shows the **rendered** special-token string — that is what the loss actually sees.


In [4]:
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

rows = []
with open("poe2_data.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

raw = Dataset.from_list(rows)


def alpaca_to_conversation(example):
    user = example["instruction"]
    if example.get("input"):
        user = f"{example['instruction']}\n\n{example['input']}"
    return {
        "conversations": [
            {"role": "user", "content": user},
            {"role": "assistant", "content": example["output"]},
        ]
    }


def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False,
        )
        for convo in examples["conversations"]
    ]
    return {"text": texts}


dataset = raw.map(alpaca_to_conversation)
dataset = dataset.map(formatting_prompts_func, batched=True)
split = dataset.train_test_split(test_size=0.2, seed=3407)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)}  Eval: {len(eval_dataset)}")
print(train_dataset[0]["conversations"])


Map: 100%|██████████| 260/260 [00:00<00:00, 15518.54 examples/s]

Train: 208  Eval: 52
[{'role': 'user', 'content': 'Provide a step-by-step PoE 2 crafting guide using the provided base item and outcome goal.\n\nCrafting Item: Omen of Connections\nCategory: Crafting Omen\nScenario:\nBase Item: 6-Socket Rare Chest Piece\nInventory State: Omen of Connections present\nTool: Orb of Fusing\nGoal: Instantly 6-link the item.'}, {'role': 'assistant', 'content': '### Path of Exile 2 Crafting Overview: Omen of Connections\n\n**Core Mechanic:** Guarantees that applying an Orb of Fusing will instantly achieve maximum linked sockets on an item.\n**Crafting Context:** Bypassing long Orb of Fusing RNG sequences to secure full socket linkage.\n\n**Step-by-Step Execution:**\nApply a single Orb of Fusing to the 6-Socket Chest Piece while holding Omen of Connections. The Omen is consumed and forces the Orb of Fusing to link all 6 sockets in a single action.'}]


### What you should have just seen

`Map` twice over 260 examples, then:

`Train: 208  Eval: 52`

and `train_dataset[0]["conversations"]` as a list of two dicts (`user`, `assistant`). If `input` was merged, the user `content` contains both the instruction and the crafting scenario.

If train/eval were 260/0, the split failed. If `conversations` still had `instruction` keys, `alpaca_to_conversation` did not run.

---

## 3b. Inspect the rendered `text` (this is the real training string)

### Why this cell exists

`conversations` is for humans. The model is trained on `text`. Printing the first ~1200 characters catches:

- **Double BOS** — `<|begin_of_text|>` twice (Unsloth later strips one; you will see a log during tokenize).
- **Missing assistant / missing `<|eot_id|>`** — the model never learns to stop.
- **Wrong template** — Vicuna `### Human:` on a Llama Instruct model.
- **System preamble** — Llama 3.1 Instruct templates often inject a knowledge-cutoff / date system turn even if you did not pass a system message. That is expected.

You are slicing `[:1200]` only so the notebook stays readable. Scroll the structure:

1. `<|begin_of_text|>` — beginning of sequence.
2. `system` header — template default.
3. `user` header + your merged instruction/input.
4. `<|eot_id|>` — user turn ends.
5. `assistant` header + gold guide.
6. `<|eot_id|>` — assistant turn ends (stop token).

If this string looks right, the trainer can be wired to the `"text"` field. If it looks wrong, **stop** and fix the template; do not train.


In [5]:
print(train_dataset[0]["text"][:1200])


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

Provide a step-by-step PoE 2 crafting guide using the provided base item and outcome goal.

Crafting Item: Omen of Connections
Category: Crafting Omen
Scenario:
Base Item: 6-Socket Rare Chest Piece
Inventory State: Omen of Connections present
Tool: Orb of Fusing
Goal: Instantly 6-link the item.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

### Path of Exile 2 Crafting Overview: Omen of Connections

**Core Mechanic:** Guarantees that applying an Orb of Fusing will instantly achieve maximum linked sockets on an item.
**Crafting Context:** Bypassing long Orb of Fusing RNG sequences to secure full socket linkage.

**Step-by-Step Execution:**
Apply a single Orb of Fusing to the 6-Socket Chest Piece while holding Omen of Connections. The Omen is consumed and forces the Orb of Fusing to link all 6 sockets in 

### What you should have just seen

A Llama 3.1 formatted transcript, including `<|start_header_id|>user/assistant<|end_header_id|>` and the crafting answer. The system block with "Cutting Knowledge Date" is the Instruct template, not your JSONL.

That `text` column is what `dataset_text_field="text"` will tokenize.

---

## 4. Build the SFT trainer (no training yet)

### Why this cell exists

**SFTTrainer** (from Hugging Face **TRL**) is the standard loop: batch strings → tokenize → forward → loss → backward → optimizer step. Unsloth patches this trainer for speed; you still configure it with **`SFTConfig`**.

This cell **constructs** the trainer and then wraps it with `train_on_responses_only`. It does **not** run GPU training. That is the next cell.

### TRL API names (so you are not confused by old blogs)

Older notebooks pass `tokenizer=` and `max_seq_length=` into `SFTTrainer`. Current TRL uses:

- `processing_class=tokenizer`
- `max_length=...` inside `SFTConfig`
- `dataset_text_field="text"` inside `SFTConfig`

Same idea, new argument names.

### `DataCollatorForSeq2Seq`

A collator pads examples in a batch to the same length. `DataCollatorForSeq2Seq` also pads **labels**. We need that because `train_on_responses_only` sets ignored positions to **`-100`**. Cross-entropy ignores index `-100`. A naive language-model collator can mess up those labels.

### Effective batch size (read this before you change numbers)

\[
\text{effective batch} = \text{per\_device\_train\_batch\_size} \times \text{gradient\_accumulation\_steps} \times \text{num\_GPUs}
\]

Here: \(2 \times 4 \times 1 = 8\).

One **optimizer step** averages 8 examples. Unsloth fixed a historical bug: `batch=1, accum=8` and `batch=8, accum=1` now match. Prefer **smaller micro-batch + more accumulation** if you OOM.

Unsloth's generic recommendation is effective batch **16** (`2 × 8`). This notebook uses **8** — slightly noisier updates, fine for 208 rows. Steps per epoch = \(\lceil 208 / 8 \rceil = 26\). Two epochs ⇒ **52 steps** (matches the training log).

### `SFTConfig` arguments

**`per_device_train_batch_size=2`** — examples per GPU per micro-step. Primary VRAM knob. Raise only if you still have spare memory; padding used to make large batches *slower*, which is why Unsloth added packing / padding-free paths.

**`gradient_accumulation_steps=4`** — wait 4 micro-batches before one Adam step. Simulates a larger batch without storing 8 full activations at once.

**`warmup_steps=10`** — linearly ramp LR from 0 → `learning_rate` over 10 steps (~20% of 52, a bit richer than the usual 5–10% heuristic). Prevents the first steps from taking huge noisy updates.

**`num_train_epochs=2`** — pass over the train set twice. Unsloth: **1–3 epochs** for instruction data. More than 3 on 208 rows is how you overfit. **Do not set `max_steps` at the same time.** `max_steps` wins and you will not know which schedule you ran. The commented `max_steps=60` is a smoke test: uncomment it **and** comment out `num_train_epochs`.

**`learning_rate=2e-4`** — Unsloth default for LoRA/QLoRA SFT (`2e-4` to `5e-6` is the usual band). RL methods want ~`5e-6`. Full FT wants lower still. If loss explodes, lower LR. If loss barely moves, raise slightly or train longer.

**`logging_steps=1`** — log every step. Fine for 52 steps; noisy on 10k-step runs.

**`eval_strategy="epoch"`** — after each epoch, run the 52-row eval set. Slow on huge evals; here it is cheap. Alternative: `eval_steps=N`.

**`output_dir="outputs_ft_cursor"`** — checkpoints (`checkpoint-52`, tokenizer, adapter). Not the same folder as the final `ft_lora` export.

**`optim="adamw_8bit"`** — AdamW with 8-bit optimizer states (bitsandbytes). Big VRAM win vs fp32 Adam. Unsloth notebooks use this by default.

**`weight_decay=0.01`** — L2-style penalty on weights. Unsloth recommended starting value. Helps generalization; do not crank to 0.5.

**`lr_scheduler_type="linear"`** — after warmup, LR decays linearly to 0. `cosine` is the other common choice. Either is fine at this scale.

**`seed=3407`** — dataloader shuffle / init seed. Matches LoRA and dataset split.

**`report_to="none"`** — do not stream to W&B/TensorBoard. Set `"wandb"` if you want charts.

**`dataset_text_field="text"`** — which column to tokenize. Must exist (we created it).

**`max_length=max_seq_length`** — 2048. Must match the loaded model context.

**`packing=False`** — **packing** concatenates short examples into one 2048-token stream. Faster, but **row count shrinks** and loss numbers are not comparable. Unsloth: with `packing=False` they still use padding-free batching, so you are not leaving free speed on the table the old way. This dataset is 260 short guides — keep `False` so 208 examples stay 208 examples.

### `train_on_responses_only(trainer)`

Default causal LM loss trains on **every** token, including the user question. Then the model spends capacity copying the prompt.

The QLoRA paper found **masking the prompt** and training only on assistant tokens gains ~1% and is especially important for chat. Unsloth implements that by setting label ids to `-100` on the user (and system) spans.

The docs show explicit markers for Llama 3.x:

```python
instruction_part = "<|start_header_id|>user<|end_header_id|>\\n\\n"
response_part    = "<|start_header_id|>assistant<|end_header_id|>\\n\\n"
```

This notebook calls `train_on_responses_only(trainer)` **without** those strings. Current Unsloth **auto-detects** them from the template. The log should look like:

`Auto-detected instruction_part = '<|start_header_id|>user...' and response_part = '<|start_header_id|>assistant...'`

If you ever see **"All labels are -100"** / loss always 0, detection failed — pass the two strings explicitly (see Unsloth troubleshooting).

You will also see **double BOS removed** and tokenization of 208 + 52 rows. That is still setup, not `trainer.train()`.


In [6]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        # max_steps=60,  # smoke test only; unset num_train_epochs if you enable this
        learning_rate=2e-4,
        logging_steps=1,
        eval_strategy="epoch",
        output_dir="outputs_ft_cursor",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
        dataset_text_field="text",
        max_length=max_seq_length,
        packing=False,
    ),
)

trainer = train_on_responses_only(trainer)


Unsloth: reducing dataset_num_proc 6 -> 4 to fit free memory (~1GB per worker). Set UNSLOTH_DATASET_NUM_PROC to override.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=4): 100%|██████████| 208/208 [00:01<00:00, 143.43 examples/s]


Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=4): 100%|██████████| 52/52 [00:01<00:00, 39.60 examples/s]


Unsloth: Auto-detected instruction_part = '<|start_header_id|>user<|end_header_id|>\n\n' and response_part = '<|start_header_id|>assistant<|end_header_id|>\n\n'


Map: 100%|██████████| 52/52 [00:00<00:00, 13247.32 examples/s]


### What you should have just seen

- Tokenizing `text` for train (208) and eval (52).
- Possible `reducing dataset_num_proc` (tokenizing subprocesses vs RAM).
- Double-BOS cleanup.
- Auto-detected Llama 3 user/assistant splitters.
- Extra `Map` passes that attach masked `labels`.

The trainer object is ready. No weights have been updated yet.

---

## 5. Run training (`trainer.train()`)

### Why this cell exists

This is the actual GPU loop. Each **step**:

1. Take the next micro-batch of tokenized `text`.
2. Forward pass (4-bit base + LoRA).
3. Loss = cross-entropy on tokens where `labels ≠ -100` (assistant only).
4. Backward through LoRA (and checkpointed activations).
5. Every `gradient_accumulation_steps` micro-batches: clip/apply **AdamW 8-bit**, step the LR scheduler.
6. Periodically eval, log, checkpoint.

`trainer_stats = trainer.train()` returns a `TrainOutput` (global step, average loss, timing). Displaying `trainer_stats` prints that object.

### How to read the Unsloth training banner

From a previous successful run on this notebook:

| Field | Example | Meaning |
| --- | --- | --- |
| Num examples | 208 | Train rows |
| Num Epochs | 2 | Full passes |
| Total steps | 52 | \(26 \times 2\) |
| Batch size per device | 2 | Micro-batch |
| Gradient accumulation | 4 | |
| Total batch size | 8 | Effective batch |
| Trainable parameters | 41.9M / 8.07B (0.52%) | LoRA-only |

`smartly offload gradients` / `Double buffering` — Unsloth VRAM tricks. Harmless.

### What is a "good" loss?

Unsloth: many SFT runs land around **0.5–1.0**. This run's `train_loss ≈ 0.80` is in that band.

| Loss behavior | Likely meaning |
| --- | --- |
| Smoothly falling toward ~0.5–1.0 | Healthy |
| Flat / not decreasing | LR too low, bug in labels, or data too easy/random |
| Drops toward **0** | Overfitting (memorizing 208 rows) |
| Explodes / NaN | LR too high, bad numerical setup |

Eval loss (logged each epoch) should **not** shoot up while train loss keeps falling — that is the overfitting signature.

### Overfit vs underfit (Unsloth's practical advice)

**Overfit** (too specialized): cut epochs, raise `weight_decay`, maybe `lora_dropout=0.1`, more diverse data, or scale `lora_alpha` down at inference.

**Underfit** (still generic): slightly higher LR or rank, more epochs (carefully), more on-domain data, smaller batch so updates are stronger.

With 260 rows, **overfitting is the default risk**, not underfitting. Two epochs is already a full meal.

### Time

~106 seconds for 52 steps on this GPU is the right order of magnitude. If it is 10× slower, you are on CPU or CUDA is not used.

When the cell finishes, you have a finetuned adapter **in memory**. Next we generate, then we save (RAM is not a checkpoint).


In [7]:
trainer_stats = trainer.train()
trainer_stats


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 208 | Num Epochs = 2 | Total steps = 52
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,0.327755,0.389760
2,0.005258,0.017045


Filter: 100%|██████████| 52/52 [00:00<00:00, 13884.89 examples/s]
Unsloth: Restored added_tokens_decoder metadata in outputs_ft_cursor/checkpoint-52/tokenizer_config.json.


TrainOutput(global_step=52, training_loss=0.7994460738801326, metrics={'train_runtime': 105.9634, 'train_samples_per_second': 3.926, 'train_steps_per_second': 0.491, 'total_flos': 4302167407902720.0, 'train_loss': 0.7994460738801326, 'epoch': 2.0})

### What you should have just seen

`TrainOutput(global_step=52, training_loss≈0.80, epoch=2.0)` plus runtime metrics (`train_samples_per_second`, `total_flos`, …). A checkpoint under `outputs_ft_cursor/checkpoint-52` is written by the trainer.

If `global_step` is 60, you still had `max_steps=60` enabled. If loss is 0.0, response masking is wrong.

---

## 6. Inference: talk to the finetuned model

### Why this cell exists

Loss is not a PoE 2 exam. You must **read an answer**. This cell:

1. Switches the model from training mode to Unsloth's fast inference path.
2. Builds a **user-only** chat (no gold assistant).
3. Generates tokens on CUDA while streaming text to the notebook.

### `FastLanguageModel.for_inference(model)`

Unsloth: always call this before `generate`. It enables their **2× inference** path (QLoRA, LoRA, and dense). Training-specific kernels (checkpointing, dropout) are not what you want at decode time.

### Building `messages`

```python
[{"role": "user", "content": "What is the best way to craft quarterstaff with elemental damage focus?"}]
```

This prompt is **not** copied from a train row on purpose. You want a sniff test of generalization. One prompt is not an eval set — it is a sanity check.

### `apply_chat_template(..., tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True)`

Opposite of training:

- **`tokenize=True`** — return tensors of ids, not a Python string.
- **`add_generation_prompt=True`** — append the assistant header so the next tokens are the model's reply.
- **`return_tensors="pt"`** — PyTorch tensors.
- **`return_dict=True`** — dict with `input_ids` (and attention mask), unpackable as `**inputs`.

`.to("cuda")` moves each tensor to the GPU. Forgetting this causes CPU/GPU device errors.

### `TextStreamer(tokenizer, skip_prompt=True)`

Prints tokens as they are generated. `skip_prompt=True` hides the echoed user template so you only see the answer.

### `model.generate` arguments

**`max_new_tokens=512`** — cap on the reply length. Unsloth: raise this if answers cut off (256, 1024, …). You wait longer.

**`use_cache=True`** — KV cache. Standard; do not turn off unless debugging memory.

**`temperature=1.5`** — sampling softness. `1.0` is the model's raw distribution. **Higher = more random.** Unsloth conversational demos often use 1.5 for lively chat. For a **crafting guide**, you may prefer `0.3–0.7` so recipes are less whimsical. This notebook leaves 1.5 as the Unsloth-style demo.

**`min_p=0.1`** — min-p sampling: drop tokens with probability below 10% of the top token. A modern alternative to top-p. With high temperature, min-p stops the tail from producing garbage.

The warning `max_new_tokens and max_length seem to have been set` is Transformers noise: the tokenizer/model config may still advertise a huge `max_length` (e.g. 131072). **`max_new_tokens` wins.** Ignore it.

### How to judge the printed answer

Ask:

- Does it use the **guide structure** from the dataset (core mechanic, context, steps)?
- Does it sound like PoE 2, or generic RPG filler?
- Does it stop at `<|eot_id|>` or ramble?

A pretty train loss with a useless sample means the dataset does not cover that question (or you overfit to template phrases). Fix data, not `r`.

After this, the adapter still lives only in this Python process until you save.


In [8]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "user",
        "content": "What is the best way to craft quarterstaff with elemental damage focus?",
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)
inputs = {key: value.to("cuda") for key, value in inputs.items()}

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    use_cache=True,
    temperature=1.5,
    min_p=0.1,
)


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Here is the breakdown for crafting Quarterstaff with elemental damage focus:

**Base Item Analysis:** Quarterstaff
**Type:** Bludgeoning weapon used for melee combat at range.
**Crafting Context:** Elemental damage dealers seeking a versatile, easy-to-manage reach weapon.

**Stat Priority:** Force, Lightning, Frost, or Poison damage affixes.
**Crafting Procedure:** Apply elemental damage prefixes or suffixes to the quarterstaff via randomized crafting or custom enhancements.
**Tips & Variations:** Bypassing melee distance restrictions while dealing elemental damage from a safe range.<|eot_id|>


### What you should have just seen

A streamed crafting-style answer ending at `<|eot_id|>`. Quality varies by prompt. Re-run generate with a train-like instruction if you want a "did it memorize the format?" check vs a held-out question.

---

## 7. Save LoRA adapters (do this; it is the real artifact)

### Why this cell exists

`trainer.train()` checkpoints under `outputs_ft_cursor/`, but the clean delivery artifact is a **small LoRA folder** you can reload later:

```python
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
```

That writes (among other files):

| File | Role |
| --- | --- |
| `adapter_config.json` | Rank, alpha, target modules, base model id |
| `adapter_model.safetensors` | Trained \(A,B\) weights (tens to hundreds of MB, not 16 GB) |
| tokenizer files + `chat_template.jinja` | How to encode/decode the same way as training |

**You did not save a full 8B model.** At load time you still need the **same base** (`unsloth/Meta-Llama-3.1-8B-Instruct-unsloth-bnb-4bit` or the 16-bit Instruct original) plus this adapter.

Reload pattern (later session):

```python
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="ft_lora",  # adapter dir; Unsloth reads adapter_config for the base
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

Use the **same chat template** at inference as at train. Unsloth's #1 export bug: Unsloth notebook looks great, Ollama looks drunk — almost always template / EOS mismatch.

Hugging Face upload is `model.push_to_hub(...)` / `tokenizer.push_to_hub(...)` if you want a private repo; not required here.

`lora_dir = "ft_lora"` is a relative path (project folder). The print confirms the path. The "Restored added_tokens_decoder metadata" line is Unsloth fixing tokenizer JSON so reload stays consistent.


In [9]:
lora_dir = "ft_lora"
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f"Saved LoRA adapters to {lora_dir}")


Unsloth: Restored added_tokens_decoder metadata in ft_lora/tokenizer_config.json.


Saved LoRA adapters to ft_lora


### What you should have just seen

`Saved LoRA adapters to ft_lora`. That folder is enough to resume training or run Unsloth inference. GGUF is only for **other** runtimes.

---

## 8. Optional: merge + export GGUF (llama.cpp / Ollama)

### Why this cell exists

LoRA adapters are not what [llama.cpp](https://github.com/ggml-org/llama.cpp), **Ollama**, or LM Studio load. Those want a **GGUF** file: frozen weights + your LoRA **merged**, then quantized.

```python
model.save_pretrained_gguf(
    "ft_cursor_gguf",
    tokenizer,
    quantization_method="q4_k_m",
    maximum_memory_usage=0.75,
)
```

Unsloth merges adapters into the dense model, converts, and quantizes. It is **slow** and **VRAM-heavy**. This cell is gated with `if False:` so a full notebook "Run All" does not stall for a merge you did not ask for. Flip to `if True:` when you need GGUF.

### Arguments

**`"ft_cursor_gguf"`** — output directory / prefix for the `.gguf` file(s).

**`quantization_method="q4_k_m"`** — Unsloth's recommended everyday quant: mixed 4-bit K-quants, Q6_K on some attention/FFN tensors. Smaller and faster than F16, much better than ancient `q4_0`. Other useful values from the [GGUF docs](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf):

| Method | When |
| --- | --- |
| `q4_k_m` | Default deploy |
| `q5_k_m` | Extra quality, larger |
| `q8_0` | Heavy, closer to fp16 |
| `f16` | Archival / conversion, huge |

**`maximum_memory_usage=0.75`** — cap peak GPU use during save at 75% of VRAM. If merge **OOM**s, drop to `0.5` or `0.4`.

### After export

Point Ollama / llama.cpp at the GGUF **and** use Llama 3.1 Instruct chat formatting. If Unsloth streaming looked good and GGUF looks broken, fix the template, not the rank.

You can also `push_to_hub_gguf("username/repo", tokenizer, quantization_method="q4_k_m")`.

Manual path if Unsloth GGUF fails: `save_pretrained_merged(..., save_method="merged_16bit")` then `convert_hf_to_gguf.py` from llama.cpp.

Leave this cell `False` until adapters in `ft_lora` are answers you are willing to ship.


In [10]:
if False:
    model.save_pretrained_gguf(
        "ft_cursor_gguf",
        tokenizer,
        quantization_method="q4_k_m",
        maximum_memory_usage=0.75,
    )


## Course wrap-up: what you actually did

You ran a complete **QLoRA SFT** loop:

1. Loaded a **dynamic 4-bit Llama 3.1 8B Instruct** with Unsloth (`max_seq_length=2048`, `dtype` auto-bf16).
2. Attached **LoRA r=16** on all attention + MLP projections (~0.52% of weights).
3. Mapped **Alpaca JSONL** → Llama 3.1 **chat `text`** (not a live formatting function).
4. Held out **20%** eval (208 / 52).
5. Trained with **TRL SFTTrainer**, effective batch **8**, **2 epochs**, **52 steps**, **LR 2e-4**, loss **on assistant tokens only**, **packing off**.
6. Generated with **`for_inference`** + chat template **`add_generation_prompt=True`**.
7. Saved **LoRA** to `ft_lora`. GGUF remains optional.

### The ideas to remember when you start a new project

- **Data > rank.** r=16 and 2e-4 are strong defaults. Garbage JSONL cannot be rescued by r=128.
- **Template must match** at train, Unsloth generate, and Ollama.
- **QLoRA first**, full FT last.
- **`num_train_epochs` XOR `max_steps`**, never both.
- **Mask user tokens** (`train_on_responses_only`) for chat SFT.
- **Save adapters immediately**; a crashed kernel drops in-memory weights.

### Sensible next experiments (change one thing at a time)

- Lower generate `temperature` to 0.4 and compare crafting consistency.
- Add a real qualitative eval: 10 held-out scenarios scored by you.
- Grow `poe2_data.jsonl` with harder crafts before you touch `r`.
- Try `lora_alpha=32` if the model still sounds like stock Llama.
- Stop at 1 epoch if eval loss got worse at epoch 2.

If a later notebook diverges from [Unsloth's fine-tuning guide](https://unsloth.ai/docs/get-started/fine-tuning-llms-guide), trust the guide for new API names, and trust **your printed `text` sample** for whether the dataset is sane.
